# 02 満足度と公平性

- 分担 $(x, y)$ から、**2 人それぞれの満足度**を計算する
- **平均満足度・最低満足度・満足度の差**を区別する（平均だけで公平性を評価しない）
- **公平性の定義**を変えると、同じ分担の評価がどう変わるかを調べる（RQ1・RQ2）
- 計算：`household_model.py`・`evaluation.py`

> **教育用の架空データ・架空モデル**です。Partner A・B は実在の人物ではなく、性別や役割を意味しません。満足度関数や関係解消確率の係数は架空の値で、実際の離婚率や特定の関係の将来を予測するものではありません。

## 準備：実験環境（最初に必ず実行）

F4 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F4 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F4 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F4 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                 # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f4-partnership-simulation"     # リポジトリの中の F4 フォルダ（ルートが F4 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f4_root(p):
    return (Path(p) / "household_model.py").exists() and (Path(p) / "relationship_dynamics.py").exists()


def find_f4_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F4 のルート
        if is_f4_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f4_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F4 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f4_upload")
    for hit in glob.glob("/content/f4_upload/**/household_model.py", recursive=True):
        if is_f4_root(Path(hit).parent):
            return Path(hit).parent
    return None


F4_ROOT = find_f4_root()
if F4_ROOT is None:
    raise RuntimeError("F4 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F4_ROOT)
if str(F4_ROOT) not in sys.path:
    sys.path.insert(0, str(F4_ROOT))
if not (F4_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。decision_strategies.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import partners as pt
import household_model as hm
import decision_strategies as ds
import relationship_dynamics as rd
import evaluation as ev
import partnership_charts as pc

pc.setup_style()                       # 文字を大きくするなら pc.setup_style(1.3)
scenarios = pt.load_scenarios()        # シナリオ A〜D（すべて架空）
params = rd.load_parameters()          # モデルのパラメータ（すべて架空）
events = pt.load_stress_events()       # ストレスイベントの組
SC = pt.scenario_by_id(scenarios, "A") # 基準にする初期条件（Scenario A）
P, HH = SC["partners"], SC["household"]
COND0 = dict(HH, stress=params["base_stress"])   # ストレスイベントのない年の生活条件
CANDS = ds.candidate_grid(params["share_step"])  # 分担の候補（全戦略で共通）
SEEDS = range(5)                       # 乱数シード
REPLICATES = 200                       # シードごとの反復回数
YEARS = params["simulation_years"]

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        print(ev.format_table(rows, columns))


print("F4_ROOT:", F4_ROOT)
print("シナリオ:", [s["label"] for s in scenarios])
print("モデルのパラメータ（架空）:", {k: params[k] for k in ("alpha", "baseline_hazard", "beta", "gamma",
                                                       "base_stress", "fairness_lambda", "share_step", "simulation_years")})

## 1. 満足度関数（教育用の重み付きモデル）

$$
U_i = \mathrm{clip}\Big(\text{base} - (1 - b\,c)\,(w_1 \cdot \text{家事のずれ}_i + w_2 \cdot \text{生活費のずれ}_i)
+ w_3\,c - w_4 \cdot \text{Stress} + w_5 \cdot \text{自由時間}_i - w_6 \cdot \text{超過}_i
- w_7 f_i \cdot \text{不公平感}_i - w_8 \cdot \text{家計の圧迫}_i,\ 0,\ 1\Big)
$$

| 要素 | 意味（0〜1） |
| --- | --- |
| 家事・生活費のずれ | \|実際の分担 − 希望の分担\| |
| $c$ | 2 人のコミュニケーション指標の平均。$b$ = `communication_buffer`（話し合いで不満が和らぐ程度） |
| 自由時間 | (70 − 労働時間 − 家事時間) ÷ 70 |
| 超過 | 家事に使える時間・所得を超えた分 |
| 不公平感 | 自分の（労働 + 家事）時間が相手より多い分 ÷ 20 時間。$f_i$ = 公平性を重視する程度 |
| 家計の圧迫 | 生活費の負担 ÷ 自分の所得 が 0.5 を超えた分 |

**係数はすべて架空の値**です。実際の夫婦関係の満足度を予測するものではありません。

In [ ]:
x, y = 0.5, 0.5
rows = []
for pid in pt.PARTNERS:
    c = hm.utility_components(pid, x, y, P, COND0, params)
    rows.append({"partner": pid, **c, "U": hm.utility(pid, x, y, P, COND0, params)})
table(rows, [("partner", "partner", "{}")] + [(k, k, "{:.3f}") for k in hm.UTILITY_KEYS] + [("U", "U (satisfaction)", "{:.3f}")])
print("重み:", {k: params[k] for k in ("base_satisfaction", "w_housework", "w_financial", "w_communication",
                                    "w_stress", "w_free_time", "w_overload", "w_fairness", "w_financial_strain")})

## 2. 分担率と満足度

生活費を 50:50 に固定して家事分担率 $x$ を動かすと、2 人の満足度はどう変わるか。

In [ ]:
#@title 家事分担率と満足度
xs = ds.share_grid(0.05)
ua = [hm.utility("A", x, 0.5, P, COND0, params) for x in xs]
ub = [hm.utility("B", x, 0.5, P, COND0, params) for x in xs]
marks = {"50:50": 0.5, "work hours": hm.work_hours_housework_share(P), "preference": hm.preference_housework_share(P)}
show(pc.plot_share_sweep(xs, ua, ub, "x = Partner A's housework share (expenses 50:50)",
                         "Housework share and satisfaction (year 1, no event)", marks), "02_sweep_housework")

In [ ]:
#@title 生活費の負担率と満足度
ys = ds.share_grid(0.05)
ua = [hm.utility("A", 0.5, y, P, COND0, params) for y in ys]
ub = [hm.utility("B", 0.5, y, P, COND0, params) for y in ys]
marks = {"50:50": 0.5, "income": hm.income_share(P), "preference": hm.preference_financial_share(P)}
show(pc.plot_share_sweep(ys, ua, ub, "y = Partner A's expense share (housework 50:50)",
                         "Expense share and satisfaction (year 1, no event)", marks), "02_sweep_expenses")

## 3. 2 人の満足度の比較

平均が同じでも、**どちらか一方だけが低い**ことがあります。平均・最低・差を並べて見ます。

In [ ]:
#@title 2 人の満足度の比較
RULES = {"50:50 / 50:50": (0.5, 0.5),
         "work hours / income": (hm.work_hours_housework_share(P), hm.income_share(P)),
         "80:20 / 50:50": (0.8, 0.5),
         "preference / preference": (hm.preference_housework_share(P), hm.preference_financial_share(P))}
rule_rows = []
for name, (x, y) in RULES.items():
    a, b = hm.utilities(x, y, P, COND0, params)
    rule_rows.append({"name": name, "x": x, "y": y, "A": a, "B": b, "mean": ev.mean_satisfaction(a, b),
                      "min": ev.minimum_satisfaction(a, b), "diff": ev.satisfaction_difference(a, b)})
table(rule_rows, [("name", "housework / expenses", "{}"), ("x", "x", "{:.2f}"), ("y", "y", "{:.2f}"),
                  ("A", "U_A", "{:.3f}"), ("B", "U_B", "{:.3f}"), ("mean", "mean", "{:.3f}"),
                  ("min", "min", "{:.3f}"), ("diff", "diff", "{:.3f}")])
show(pc.plot_partner_bars(rule_rows, "Satisfaction of each partner by sharing rule (year 1)"), "02_partner_bars")

## 4. 公平性指標（教育用）

$$ \text{Fairness} = 1 - \frac{|x - x^*| + |y - y^*| + |U_A - U_B|}{3} $$

$(x^*, y^*)$ は「公平とみなす基準の分担」です。**何を基準にするかはモデル上の選択**です。

| 基準 | $x^*$（家事） | $y^*$（生活費） |
| --- | --- | --- |
| workload | 労働 + 家事 の時間が 2 人で等しい | 所得比例 |
| equal | 0.5 | 0.5 |
| income | 所得の少ない人が多く（比較用） | 所得比例 |
| preference | 2 人の希望の中間 | 2 人の希望の中間 |

In [ ]:
fair_rows = []
for r in rule_rows:
    row = {"name": r["name"]}
    for ref in ev.FAIRNESS_REFERENCES:
        row[ref] = ev.fairness_index(r["x"], r["y"], r["A"], r["B"], P, COND0, ref)
    fair_rows.append(row)
table(fair_rows, [("name", "housework / expenses", "{}")] + [(ref, "fair: " + ref, "{:.3f}") for ref in ev.FAIRNESS_REFERENCES])
for ref in ev.FAIRNESS_REFERENCES:
    order = ds.rank_strategies({r["name"]: r[ref] for r in fair_rows})
    print(f"{ref:>10}: " + " > ".join(n for n, _v in order))

In [ ]:
#@title 公平性指標のヒートマップ
grid = ds.share_grid(params["share_step"])
REF = "workload"                     # TODO：equal・income・preference に変える
values = []
for y in grid:
    row = []
    for x in grid:
        if hm.is_feasible(x, y, P, COND0):
            a, b = hm.utilities(x, y, P, COND0, params)
            row.append(ev.fairness_index(x, y, a, b, P, COND0, REF))
        else:
            row.append(None)          # 実行不可能（白）
    values.append(row)
g = ds.greedy_allocation(P, COND0, params, CANDS)
f = ds.fairness_allocation(P, COND0, params, CANDS)
show(pc.plot_heatmap(grid, grid, values, f"Fairness index (reference: {REF}); blank = infeasible", "fairness index",
                     {"50:50": (0.5, 0.5), "max mean satisfaction (Greedy)": g, "fairness-aware choice": f}),
     "02_fairness_heatmap")

## 5. 学生向け TODO：重みと公平性の定義を変える

パラメータのコピーを変えて、**平均満足度が最大になる分担**と**公平性つき目的関数が最大になる分担**がどう変わるか調べます（元のデータは変更しません）。

In [ ]:
# TODO 1：重みを変えて実行しよう
p2 = rd.with_changes(params, w_housework=0.3, w_fairness=0.6)
for name, prm in [("元の重み", params), ("変えた重み", p2)]:
    print(f"{name}: 平均満足度が最大 {ds.greedy_allocation(P, COND0, prm, CANDS)}"
          f" / 公平性つき {ds.fairness_allocation(P, COND0, prm, CANDS)}")

# TODO 2：上のセルの REF を変えて、ヒートマップの明るい場所がどう動くか見る
# TODO 3：fairness_index の weights（家事・生活費・満足度差の重み）を変えて、順位がどう変わるか

## 考察（チームで記入）

- **TODO**：平均満足度が最も高い分担と、最低満足度が最も高い分担は同じか
- **TODO**：公平性の基準を変えると、どの分担の評価が大きく変わったか。それはなぜか（RQ1）
- **TODO**：生活費の 50:50・所得比例・希望の中間は、2 人の満足度にどう違う影響を与えたか（RQ2）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 単純挿入ソート）
- F4 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_partnerships.json`（この教材のための**架空のデータ**）